# Treino do modelo baseline e comparação

**Dataset:** [Telco Customer Churn: IBM dataset](https://www.kaggle.com/datasets/yeanzc/telco-customer-churn-ibm-dataset) (fonte: Kaggle).

**Objetivo final:** Realizar o treino do modelo baseline e de modelos desafiantes de forma a comparar seus resultados.

### 1. Preparação do dataset (teste)

**Objetivo desta etapa**: Definição das bibliotecas, importação do dataset e definição de parâmetros globais para treino.

In [106]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Uma homenagem ao "Guia do Mochileiro das Galáxias", de Douglas Adams
RANDOM_STATE = 42
TEST_SIZE = 0.2

df = pd.read_excel("Telco_customer_churn.xlsx")

**Objetivo desta etapa**: Remoção de colunas irrelevantes e definição de features e target.

As colunas abaixo precisam ser retiradas do dataset de treino por quatro motivos: i. porque não representam variáveis explicativas reais (p.e. "Customer ID" e "Count"); ii. porque já são capturadas por outras variáveis (p.e. "Latitude" e "Longitute"); iii. porque tem correlação forte com outras variáveis (p.e. "Total Charge"); ou iv. porque são variáveis futuras (p.e. "Churn Score" e "Churn Reason").

In [107]:
drop_cols = ["Country","State","CustomerID","Count","Lat Long","Latitude","Longitude",
             "Total Charges","Churn Label","Churn Score","Churn Reason","CLTV"]
df_treated = df.drop(columns=drop_cols)

# Definição das variáveis explicativas (features) e da variável resposta (target)
X_original = df_treated.drop("Churn Value", axis=1)
y_original = df_treated["Churn Value"]

X_original.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   City               7043 non-null   str    
 1   Zip Code           7043 non-null   int64  
 2   Gender             7043 non-null   str    
 3   Senior Citizen     7043 non-null   str    
 4   Partner            7043 non-null   str    
 5   Dependents         7043 non-null   str    
 6   Tenure Months      7043 non-null   int64  
 7   Phone Service      7043 non-null   str    
 8   Multiple Lines     7043 non-null   str    
 9   Internet Service   7043 non-null   str    
 10  Online Security    7043 non-null   str    
 11  Online Backup      7043 non-null   str    
 12  Device Protection  7043 non-null   str    
 13  Tech Support       7043 non-null   str    
 14  Streaming TV       7043 non-null   str    
 15  Streaming Movies   7043 non-null   str    
 16  Contract           7043 non-null   

Tratamento de valores faltantes e conversão de variáveis categóricas em numéricas

In [109]:
X_treated = X_original.copy()

# Sessão 3: tratamento de valores faltantes e conversão de variáveis categóricas em numéricas
for col in X_original.select_dtypes(include=["object"]).columns:
    X_treated[col] = X_original[col].fillna(X_original[col].mode()[0])

for col in X_original.select_dtypes(include=["int64","float64"]).columns:
    X_treated[col] = X_original[col].fillna(X_original[col].median())

cathegorical_cols = ['City', 'Zip Code', 'Gender', 'Senior Citizen', 'Partner', 'Dependents',
                     'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security',
                     'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
                     'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']

# Conversão de variáveis categóricas em numéricas (one-hot encoding)
X = pd.get_dummies(X_treated, columns=cathegorical_cols, drop_first=True)

# Normalização
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

C:\Users\rodri\AppData\Local\Temp\ipykernel_6072\654692043.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X_original.select_dtypes(include=["object"]).columns:


### 2. Treinamento de modelos

**Objetivo desta etapa**: Treinamento de um modelo baseline e dois modelos desafiantes a partir do mesmo estado inicial.

Lembrando que a natureza do problema de negócio é prever o cancelamento de um serviço, ou seja, a resposta é dicotômica (sim ou não). Por este motivo a regressão logística é a candidata ideal para modelo baseline -- isto é, aquele cujo resultado inicial será utilizado como referência nas tentativas posteriores em melhor o poder preditivo.

Testaremos três modelos ao todo.

1. Regressão logística (baseline)
2. Random Forest
3. SVM

In [112]:
# CÓDIGO ORIGINAL

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Dense

# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

# 1. Regressão Logística
log_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced")
log_model.fit(X_train, y_train)
#y_pred_log = log_model.predict(X_test)

# 2. Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,        # mais árvores para estabilidade
    max_depth=10,            # limitar profundidade
    min_samples_split=10,    # mínimo de amostras para dividir um nó
    min_samples_leaf=5,      # mínimo de amostras em uma folha
    random_state=42,
    class_weight="balanced"  # útil em dataset desbalanceado
    )
rf_model.fit(X_train, y_train)
#y_pred_rf = rf_model.predict(X_test)

# 3. SVM
svm_model = SVC(kernel="rbf", probability=True)
svm_model.fit(X_train, y_train)
#y_pred_svm = svm_model.predict(X_test)

# 4. Rede Neural
# nn_model = Sequential()
# nn_model.add(Dense(64, input_dim=X_train.shape[1], activation="relu"))
# nn_model.add(Dense(32, activation="relu"))
# nn_model.add(Dense(1, activation="sigmoid"))

# nn_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
# nn_model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=1)

# y_pred_nn = (nn_model.predict(X_test) > 0.5).astype("int32")

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


Análise qualitativa das features que entraram no modelo de regressão logística.

In [104]:
# Obter os nomes das variáveis após o One-Hot Encoding
feature_names = X.columns  # X é o DataFrame após o get_dummies

# Coeficientes da regressão logística
coeficientes = log_model.coef_[0]  # vetor de coeficientes

# Criar DataFrame com variáveis e coeficientes
coef_df = pd.DataFrame({
    "Variável": feature_names,
    "Coeficiente": coeficientes
})

# Ordenar por importância (valor absoluto do coeficiente)
coef_df["Importância"] = np.abs(coef_df["Coeficiente"])
coef_df = coef_df.sort_values(by="Importância", ascending=False)

print(coef_df.head(20))  # mostra as 20 variáveis mais relevantes

                             Variável  Coeficiente  Importância
0                       Tenure Months    -1.507909     1.507909
2784                   Dependents_Yes    -1.134014     1.134014
2803                Contract_Two year    -0.821355     0.821355
563                  City_Los Angeles     0.668723     0.668723
2788     Internet Service_Fiber optic     0.478445     0.478445
2802                Contract_One year    -0.386361     0.386361
2806  Payment Method_Electronic check     0.376325     0.376325
2797                 Tech Support_Yes    -0.301426     0.301426
2783                      Partner_Yes     0.296835     0.296835
2799                 Streaming TV_Yes     0.290763     0.290763
1313                   Zip Code_91206     0.278846     0.278846
2804            Paperless Billing_Yes     0.269183     0.269183
1160                   Zip Code_90034    -0.268691     0.268691
1971                   Zip Code_93711     0.239690     0.239690
1190                   Zip Code_90077   

### 3. Comparação de modelos

**Objetivo desta etapa**: Escolha das métricas e comparação dos modelos.

Aqui teremos uma primeira visão do desempenho inicial obtido com o modelo baseline e seus desafiantes, mas ainda sem o processo habitual de otimização que costuma fazer parte de modelos de machine learning, explicados em maior detalhe na próxima etapa. Explicação das métricas utilizadas

**Acurácia (accuracy):** Quão frequentemente o modelo está correto em geral

**Precisão (precision):** Quão frequentemente o modelo está correto quando prediz sucesso

**Sensibilidade (recall):** Qual o percentual de sucesso o modelo acertou

**F1-score:** Média harmônica de precisão e sensibilidade

In [113]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def avaliar_modelo(modelo, X_train, y_train, X_test, y_test, nome):
    # Previsões
    y_pred_train = modelo.predict(X_train)
    y_pred_test = modelo.predict(X_test)

    # Métricas treino
    acc_train = accuracy_score(y_train, y_pred_train)
    prec_train = precision_score(y_train, y_pred_train)
    rec_train = recall_score(y_train, y_pred_train)
    f1_train = f1_score(y_train, y_pred_train)

    # Métricas teste
    acc_test = accuracy_score(y_test, y_pred_test)
    prec_test = precision_score(y_test, y_pred_test)
    rec_test = recall_score(y_test, y_pred_test)
    f1_test = f1_score(y_test, y_pred_test)

    # Exibir resultados
    print(f"\n=== {nome} ===")
    print("Treino -> Acurácia: {:.3f}, Precisão: {:.3f}, Recall: {:.3f}, F1: {:.3f}".format(
        acc_train, prec_train, rec_train, f1_train))
    print("Teste  -> Acurácia: {:.3f}, Precisão: {:.3f}, Recall: {:.3f}, F1: {:.3f}".format(
        acc_test, prec_test, rec_test, f1_test))

# Avaliar cada modelo
avaliar_modelo(log_model, X_train, y_train, X_test, y_test, "Regressão Logística")
avaliar_modelo(rf_model, X_train, y_train, X_test, y_test, "Random Forest")
avaliar_modelo(svm_model, X_train, y_train, X_test, y_test, "SVM")

# Rede Neural precisa de previsões binárias
#y_pred_train_nn = (nn_model.predict(X_train) > 0.5).astype("int32")
#y_pred_test_nn = (nn_model.predict(X_test) > 0.5).astype("int32")

# print("\n=== Rede Neural ===")
# print("Treino -> Acurácia: {:.3f}, Precisão: {:.3f}, Recall: {:.3f}, F1: {:.3f}".format(
#     accuracy_score(y_train, y_pred_train_nn),
#     precision_score(y_train, y_pred_train_nn),
#     recall_score(y_train, y_pred_train_nn),
#     f1_score(y_train, y_pred_train_nn)
# ))
# print("Teste  -> Acurácia: {:.3f}, Precisão: {:.3f}, Recall: {:.3f}, F1: {:.3f}".format(
#     accuracy_score(y_test, y_pred_test_nn),
#     precision_score(y_test, y_pred_test_nn),
#     recall_score(y_test, y_pred_test_nn),
#     f1_score(y_test, y_pred_test_nn)
# ))


=== Regressão Logística ===
Treino -> Acurácia: 0.893, Precisão: 0.723, Recall: 0.955, F1: 0.823
Teste  -> Acurácia: 0.710, Precisão: 0.490, Recall: 0.527, F1: 0.508

=== Random Forest ===
Treino -> Acurácia: 0.723, Precisão: 0.482, Recall: 0.856, F1: 0.617
Teste  -> Acurácia: 0.720, Precisão: 0.504, Recall: 0.858, F1: 0.635

=== SVM ===
Treino -> Acurácia: 0.802, Precisão: 0.856, Recall: 0.291, F1: 0.434
Teste  -> Acurácia: 0.712, Precisão: 0.475, Recall: 0.145, F1: 0.222


### 4. Feature engineering

 **Objetivo desta etapa:** Aumento do poder preditivo através da aplicação de transformações às variáveis originais.

A engenharia de features (ou feature engineering) é uma etapa no qual aplicamos transformações às variáveis originais em busca de correlações não-óbvias, para então tentar aumentar o poder preditivo do modelo. Importante notar que esta etapa pode trazer benefícios para todos os métodos testados.

Resultados atualizados a partir da etapa de feature engineering.

### 5. Ajustes de hiperparâmetros

**Objetivo desta etapa:** Utilizar métodos automatizados para escolher o melhor conjunto de parâmetros.

Os ajutes de hiperparâmetros é uma etapa na qual aplicamos ajustes finos nos parâmetros iniciais, utilizados principalmente em métodos baseados em árvores de decisão, como número máximo de árvores, profundidade, número mínimo de amostras em um nó-folha etc., para então tentar aumentar um pouco mais o poder preditivo do modelo.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Espaço de busca de hiperparâmetros
param_grid = {
    "n_estimators": [100, 200, 300],        # número de árvores
    "max_depth": [5, 10, 20, None],         # profundidade máxima
    "min_samples_split": [2, 5, 10],        # mínimo de amostras para dividir um nó
    "min_samples_leaf": [1, 2, 5],          # mínimo de amostras em uma folha
    "class_weight": ["balanced"]            # ajusta pesos para lidar com desbalanceamento
}

# Modelo base
rf_base = RandomForestClassifier(random_state=42)

# GridSearch com validação cruzada
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,                # 5-fold cross-validation
    scoring="f1",        # otimizar pelo F1-score (mais adequado em churn)
    n_jobs=-1,           # usar todos os núcleos disponíveis
    verbose=2
)

# Treinar
grid_search.fit(X_train, y_train)

# Melhor combinação de parâmetros
print("Melhores parâmetros encontrados:", grid_search.best_params_)

# Modelo final com melhores parâmetros
rf_best = grid_search.best_estimator_

# Avaliação em treino e teste
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def avaliar_modelo(modelo, X_train, y_train, X_test, y_test, nome):
    y_pred_train = modelo.predict(X_train)
    y_pred_test = modelo.predict(X_test)

    print(f"\n=== {nome} ===")
    print("Treino -> Acurácia: {:.3f}, Precisão: {:.3f}, Recall: {:.3f}, F1: {:.3f}".format(
        accuracy_score(y_train, y_pred_train),
        precision_score(y_train, y_pred_train),
        recall_score(y_train, y_pred_train),
        f1_score(y_train, y_pred_train)
    ))
    print("Teste  -> Acurácia: {:.3f}, Precisão: {:.3f}, Recall: {:.3f}, F1: {:.3f}".format(
        accuracy_score(y_test, y_pred_test),
        precision_score(y_test, y_pred_test),
        recall_score(y_test, y_pred_test),
        f1_score(y_test, y_pred_test)
    ))

avaliar_modelo(rf_best, X_train, y_train, X_test, y_test, "Random Forest (GridSearchCV)")

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Melhores parâmetros encontrados: {'class_weight': 'balanced', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 200}

=== Random Forest (GridSearchCV) ===
Treino -> Acurácia: 0.987, Precisão: 0.960, Recall: 0.990, F1: 0.975
Teste  -> Acurácia: 0.793, Precisão: 0.621, Recall: 0.695, F1: 0.656
